## Notebook19a

In [1]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

In [2]:
import numpy as np
import polars as pl

from funs import *
from plotnine import *
from polars import col as c
theme_set(theme_minimal())
pl.Config.set_fmt_str_lengths(1000)

ub = "https://raw.githubusercontent.com/taylor-arnold/fds-py-nb/refs/heads/main/"

### Small Example

Much of the code itself is already included in this notebook. We are going to go through it slowly and intentionally to understand how to parse text annotations in Python. To start, load the small English spaCy model:

In [3]:
import spacy
nlp = spacy.load("en_core_web_sm")

Next, create a corpus object with a single text. I've filled in one example, but you should use the string that you had for homework.

In [12]:
docs = pl.DataFrame({
    "doc_id": "Example",
    "text": "There was once a boy named Milo who didn't know what to do with himself -- not just sometimes, but always."
})

Now parse the text with the NLP model.

In [13]:
anno = DSText.process(docs, nlp)
print_rows(anno)

doc_id,sid,tid,token,token_with_ws,lemma,upos,tag,is_alpha,is_stop,is_punct,dep,head_idx,ent_type,ent_iob
str,i64,i64,str,str,str,str,str,bool,bool,bool,str,i64,str,str
"""Example""",1,1,"""There""","""There ""","""there""","""PRON""","""EX""",true,true,false,"""expl""",2,"""""","""O"""
"""Example""",1,2,"""was""","""was ""","""be""","""VERB""","""VBD""",true,true,false,"""ROOT""",2,"""""","""O"""
"""Example""",1,3,"""once""","""once ""","""once""","""ADV""","""RB""",true,true,false,"""advmod""",2,"""""","""O"""
"""Example""",1,4,"""a""","""a ""","""a""","""DET""","""DT""",true,true,false,"""det""",5,"""""","""O"""
"""Example""",1,5,"""boy""","""boy ""","""boy""","""NOUN""","""NN""",true,false,false,"""attr""",2,"""""","""O"""
"""Example""",1,6,"""named""","""named ""","""name""","""VERB""","""VBN""",true,false,false,"""acl""",5,"""""","""O"""
"""Example""",1,7,"""Milo""","""Milo ""","""Milo""","""PROPN""","""NNP""",true,false,false,"""oprd""",6,"""PERSON""","""B"""
"""Example""",1,8,"""who""","""who ""","""who""","""PRON""","""WP""",true,true,false,"""nsubj""",11,"""""","""O"""
"""Example""",1,9,"""did""","""did""","""do""","""AUX""","""VBD""",true,true,false,"""aux""",11,"""""","""O"""


Compare this to what you did for the homework. Are there any differences?

### Grabbing Some Data

Next, we are going to grab some text data from Wikipedia. I've written a script that called the Wikipedia API to get the text from a page. Here is the function that takes a page name and returns the text.

In [14]:
import re
import requests
from lxml import html

def wiki_get_text(page):
    resp = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={
            "action": "parse",
            "page": page,
            "prop": "text",
            "format": "json",
        },
        headers={"User-Agent": "MyBot/1.0 (myemail@example.com)"},
    )
    raw_html = resp.json()["parse"]["text"]["*"]
    tree = html.fromstring(raw_html)
    text = " ".join(p.text_content().strip() for p in tree.xpath("//p"))
    text = re.sub(r"\[\d+\]", "", text)

    return text

We can use this inside of a call to `pl.DataFrame` to build a textual corpus object. Here is an example for the data science page. Start with this one and then come back with your own example.

In [21]:
df = pl.DataFrame({
    "doc_id": "Data science",
    "text": wiki_get_text("Data_science")
})
df

ilia_malinin = pl.DataFrame({
    "doc_id": "Ilia Malinin",
    "text": wiki_get_text("Ilia Malinin")
})
ilia_malinin

alysa_liu = pl.DataFrame({
    "doc_id": "Alysa Liu",
    "text": wiki_get_text("Alysa Liu")
})
alysa_liu

doc_id,text
str,str
"""Alysa Liu""",""" Alysa Liu (born August 8, 2005) is an American figure skater. She is the 2026 Winter Olympic champion in both women's singles and in the team event, the 2025 World champion, the 2022 World bronze medalist, the 2025–26 Grand Prix Final champion, a two-time Grand Prix medalist, a four-time Challenger Series champion, and a two-time U.S. national champion. At the junior level, Liu was the 2020 World Junior bronze medalist, the 2019–20 Junior Grand Prix Final silver medalist, a two-time Junior Grand Prix champion, and the 2018 U.S. junior national champion. In 2019, Liu, then 13, became the youngest-ever U.S. women's national champion. The following year, she became the youngest skater to win two senior national titles, the first woman to win consecutive U.S. titles since Ashley Wagner in 2012 and 2013 and the first woman to win the junior and senior titles back-to-back since Mirai Nagasu in 2008. Following a period of retirement, she won a world title at the 2025 World Championships, bec…"


Once you have the data, you can process it with the NLP annotation:

In [22]:
anno = DSText.process(ilia_malinin, nlp)
anno

anno2 = DSText.process(alysa_liu, nlp)
anno2

doc_id,sid,tid,token,token_with_ws,lemma,upos,tag,is_alpha,is_stop,is_punct,dep,head_idx,ent_type,ent_iob
str,i64,i64,str,str,str,str,str,bool,bool,bool,str,i64,str,str
"""Alysa Liu""",1,1,""" """,""" """,""" ""","""SPACE""","""_SP""",false,false,false,"""dep""",2,"""""","""O"""
"""Alysa Liu""",1,2,"""Alysa""","""Alysa ""","""Alysa""","""PROPN""","""NNP""",true,false,false,"""compound""",3,"""PERSON""","""B"""
"""Alysa Liu""",1,3,"""Liu""","""Liu ""","""Liu""","""PROPN""","""NNP""",true,false,false,"""nsubj""",11,"""PERSON""","""I"""
"""Alysa Liu""",1,4,"""(""","""(""","""(""","""PUNCT""","""-LRB-""",false,false,true,"""punct""",3,"""""","""O"""
"""Alysa Liu""",1,5,"""born""","""born ""","""bear""","""VERB""","""VBN""",true,false,false,"""acl""",3,"""""","""O"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Alysa Liu""",332,7,"""of""","""of ""","""of""","""ADP""","""IN""",true,true,false,"""prep""",6,"""""","""O"""
"""Alysa Liu""",332,8,"""manga""","""manga ""","""manga""","""PROPN""","""NNP""",true,false,false,"""pobj""",7,"""""","""O"""
"""Alysa Liu""",332,9,"""and""","""and ""","""and""","""CCONJ""","""CC""",true,true,false,"""cc""",8,"""""","""O"""


1. Now, for a little work on your end. Using the code that was in the book, find the top 10 most common nouns on the page.

In [23]:
(
    anno2
    .filter(c.upos == "NOUN")
    .group_by(c.lemma)
    .agg(count = pl.len())
    .sort(c.count, descending=True)
    .head(10)
)

lemma,count
str,u32
"""program""",66
"""point""",39
"""skate""",30
"""competition""",26
"""jump""",24
"""skater""",23
"""woman""",23
"""skating""",22
"""score""",21


2. And then find the top ten common adjectives.

In [24]:
(
    anno2
    .filter(c.upos == "ADJ")
    .group_by(c.lemma)
    .agg(count = pl.len())
    .sort(c.count, descending=True)
    .head(10)
)

lemma,count
str,u32
"""triple""",68
"""first""",43
"""short""",37
"""free""",35
"""american""",16
"""second""",15
"""good""",15
"""female""",12
"""senior""",12


Go back and choose another page and see how the outputs change. Then, wait for use to come together for the next step.

### Sheets

Next we are going to create a class dataset. Once that is finished, we will use the code below to read it in.

In [27]:
SHEET_ID = "1yyeOB8AcNvoDhSoMbjEEkaKKw6REXLV-mH4oJpg0qB8"
GID = "0"

url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
url

class_df = pl.read_csv(url)
class_df

state,name,page,party
str,str,str,str
"""Alabama""","""Tommy Tuberville""","""Tommy_Tuberville""","""Republican"""
"""Alabama""","""Katie Britt""","""Katie_Britt""","""Republican"""
"""Alaska""","""Lisa Murkowski""","""Lisa_Murkowski""","""Republican"""
"""Alaska""","""Dan Sullivan""","""Dan_Sullivan_(U.S._senator)""","""Republican"""
"""Arizona""","""Mark Kelly""","""Mark_Kelly""","""Democratic"""
…,…,…,…
"""West Virginia""","""Jim Justice""","""Jim_Justice""","""Republican"""
"""Wisconsin""","""Ron Johnson""","""Ron_Johnson""","""Republican"""
"""Wisconsin""","""Tammy Baldwin""","""Tammy_Baldwin""","""Democratic"""


Then, we can cycle through the rows of the dataset, putting together the Wikipedia data from each row, and then create a corpus object.

In [28]:

wiki_df = []
for row in class_df.iter_rows(named=True):
    wiki_df.append(pl.DataFrame({
        "doc_id": row['name'],
        "text": wiki_get_text(row['page'])
    }))

wiki_df = pl.concat(wiki_df)
wiki_df

doc_id,text
str,str
"""Tommy Tuberville""",""" Thomas Hawley Tuberville (/ˈtʌbərvɪl/; TUH-bərv-il; born September 18, 1954) is an American politician, and retired college football coach and sports broadcaster who is the senior United States senator from Alabama, a seat he has held since 2021. He is a member of the Republican Party. Before entering politics, Tuberville was the head football coach at Auburn University from 1999 to 2008. He was also the head football coach at the University of Mississippi from 1995 to 1998, Texas Tech University from 2010 to 2012, and the University of Cincinnati from 2013 to 2016. Tuberville won five national coach-of-the-year awards (AP, AFCA, Sporting News, Walter Camp, and Bear Bryant) after Auburn's 13–0 season in 2004, in which Auburn won the Southeastern Conference title and the Sugar Bowl, but was left out of the BCS National Championship Game. He earned his 100th career win in 2007. Tuberville is the only Auburn football coach to beat in-state rival Alabama six consecutive times. In 2015, he…"
"""Katie Britt""",""" Katie Elizabeth Boyd Britt (née Boyd; born February 2, 1982) is an American politician and attorney serving since 2023 as the junior United States senator from Alabama. A member of the Republican Party, Britt is the first woman to be elected to the U.S. Senate from Alabama and the youngest Republican woman to be elected to the Senate. She was president and CEO of the Business Council of Alabama from 2019 to 2021, and served as chief of staff for the previous incumbent, Richard Shelby, from 2016 to 2018. Britt was born Katie Elizabeth Boyd on February 2, 1982, to Julian and Debra Boyd in Enterprise, Alabama. During her youth she worked at her family's business. Her family lived near Fort Rucker in Dale County, Alabama. Her father owned a hardware store and later a boat dealership; her mother owned a dance studio. A graduate of Enterprise High School, Britt was a cheerleader and a valedictorian. After graduating in 2000 she studied political science at the University of Alabama. She was…"
"""Lisa Murkowski""",""" Lisa Ann Murkowski (/mərˈkaʊski/ mər-KOW-ski; born May 22, 1957) is an American attorney and politician serving as the senior United States senator from the state of Alaska, having held the seat since 2002. She is the first woman to represent Alaska in the U.S. Congress and is the Senate's second-most senior Republican woman. Murkowski became dean of Alaska's congressional delegation upon Representative Don Young's death. Murkowski is the daughter of former U.S. senator and governor of Alaska Frank Murkowski. She was appointed to the Senate by her father, who resigned his seat in 2002 to become Alaska's governor. Murkowski became the first Alaskan-born member of Congress and completed her father's unexpired Senate term, which ended in January 2005. Before her appointment to the Senate, she had been a member of the Alaska House of Representatives since 1999. Murkowski ran for and won a full term in 2004 with 48% of the vote. After losing the 2010 Republican primary to Tea Party candida…"
"""Dan Sullivan""",""" Daniel Scott Sullivan (born November 13, 1964) is an American politician, attorney, and Marine Corps veteran serving as the junior United States senator from the state of Alaska since 2015. A member of the Republican Party, Sullivan previously served as the commissioner of the Alaska Department of Natural Resources from 2010 to 2013, and as the Alaska Attorney General from 2009 to 2010. Sullivan grew up in a suburb of Cleveland, Ohio and graduated from Culver Academies in Indiana. He studied economics at Harvard University, then earned joint foreign service and Juris Doctor degrees from Georgetown University. He was on active duty for the United States Marine Corps from 1993 to 1997, 2004 to 2006, and in 2009 and 2013. Between 1997 and 1999, he clerked for judges on the United States Court of Appeals for the Ninth Circuit and the Alaska Supreme Court. He worked as 

As before, we will next create an annotation object.

In [30]:
anno = DSText.process(wiki_df, nlp)
anno

doc_id,sid,tid,token,token_with_ws,lemma,upos,tag,is_alpha,is_stop,is_punct,dep,head_idx,ent_type,ent_iob
str,i64,i64,str,str,str,str,str,bool,bool,bool,str,i64,str,str
"""Tommy Tuberville""",1,1,""" """,""" """,""" ""","""SPACE""","""_SP""",false,false,false,"""dep""",2,"""""","""O"""
"""Tommy Tuberville""",1,2,"""Thomas""","""Thomas ""","""Thomas""","""PROPN""","""NNP""",true,false,false,"""compound""",4,"""PERSON""","""B"""
"""Tommy Tuberville""",1,3,"""Hawley""","""Hawley ""","""Hawley""","""PROPN""","""NNP""",true,false,false,"""compound""",4,"""PERSON""","""I"""
"""Tommy Tuberville""",1,4,"""Tuberville""","""Tuberville ""","""Tuberville""","""PROPN""","""NNP""",true,false,false,"""nsubj""",14,"""PERSON""","""I"""
"""Tommy Tuberville""",1,5,"""(""","""(""","""(""","""PUNCT""","""-LRB-""",false,false,true,"""punct""",4,"""""","""O"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Cynthia Lummis""",184,13,"""Synod""","""Synod ""","""Synod""","""PROPN""","""NNP""",true,false,false,"""appos""",10,"""""","""O"""
"""Cynthia Lummis""",184,14,"""(""","""(""","""(""","""PUNCT""","""-LRB-""",false,false,true,"""punct""",13,"""""","""O"""
"""Cynthia Lummis""",184,15,"""LCMS""","""LCMS""","""LCMS""","""PROPN""","""NNP""",true,false,false,"""appos""",13,"""""","""O"""


3. Adapting the code show in the text, find the 8 nouns that are the most common on each page and print out the results to have a summary of the themes of each page. What patterns do you see? Are there any challenges? What else might you want to do with this data?

In [32]:
(
    anno
    .filter(c.upos == "NOUN")
    .group_by([c.doc_id, c.lemma])
    .agg(count = pl.len())
    .sort([c.doc_id, c.count], descending=[False, True])
    .group_by(c.doc_id)
    .head(8)
    .group_by(c.doc_id)
    .agg(top_adj = c.lemma.sort().str.join("; "))
    .pipe(print_rows)
)

doc_id,top_adj
str,str
"""John Boozman""","""%; bill; election; legislation; member; senator; term; vote"""
"""John Norvell""","""brother; daughter; member; newspaper; son; state; wife; year"""
"""Jim Risch""","""bill; election; governor; gun; lieutenant; senator; state; vote"""
"""Martin Heinrich""","""%; district; election; legislation; member; senator; state; term"""
"""John Kennedy""","""bill; company; law; money; people; state; treasurer; vote"""
"""Mike Lee""","""%; bill; election; government; senator; state; vote; waste"""
"""Chris Coons""","""%; coon; country; election; letter; office; senator; year"""
"""Mark Kelly""","""crew; day; gun; mission; space; station; time; wife"""
"""Ed Markey""","""%; bill; community; election; letter; member; senator; vote"""
